## Transaction Level Metrics

In [1]:
import pandas as pd

from src.backend.input_handler import input_handling
from src.backend.market_data import get_current_prices

In [2]:
filename = "real_transactions.csv"

transaction_data, tickers = input_handling(filename)

current_prices = get_current_prices(tickers)

[**********************88%*****************      ]  7 of 8 completed


In [3]:
transaction_data.merge(current_prices, how="left", on="Ticker").fillna(0.0)

,Portfolio,Transaction Date,Transaction Type,Ticker,Quantity,Transaction Price,Fee,Current Price
0,1,2023-03-10,BUY,VUAA.L,2,72.4600,0.00,148.360001
1,1,2023-04-11,BUY,VUAA.L,1,76.7000,0.00,148.360001
2,1,2023-05-10,BUY,VUAA.L,2,76.6800,0.00,148.360001
3,1,2023-06-09,BUY,VUAA.L,1,80.1800,0.00,148.360001
4,1,2023-07-14,BUY,VUAA.L,1,84.2600,0.00,148.360001
...,...,...,...,...,...,...,...,...
73,1,2026-05-12,SELL,XEON.DE,5,149.1054,2.10,150.184006
74,1,2026-06-11,SELL,XEON.DE,7,149.3430,2.14,150.184006
75,1,2026-07-10,SELL,XEON.DE,6,149.5995,2.12,150.184006
76,1,2024-08-06,BUY,SVXY,8,41.9000,2.15,62.880001


## Lot Calculation

In [16]:
import pandas as pd

from src.backend.input_handler import input_handling

In [22]:
filename = "real_transactions.csv"

transaction_data, tickers = input_handling(filename)

In [23]:
# transaction_data = transaction_data.sort_values(by="Transaction Date")

transactions_dict = transaction_data.to_dict(orient='records')
transactions_dict[0:2]

[{'Portfolio': '1',
  'Transaction Date': Timestamp('2023-03-10 00:00:00'),
  'Transaction Type': 'BUY',
  'Ticker': 'VUAA.L',
  'Quantity': 2,
  'Transaction Price': 72.46,
  'Fee': 0.0},
 {'Portfolio': '1',
  'Transaction Date': Timestamp('2023-04-11 00:00:00'),
  'Transaction Type': 'BUY',
  'Ticker': 'VUAA.L',
  'Quantity': 1,
  'Transaction Price': 76.7,
  'Fee': 0.0}]

In [15]:
lots = []

for row in transactions_dict:
    if row.get("Transaction Type") == "BUY":
        lots.append({
            "Ticker": row.get("Ticker"),
            "Buy Date": row.get("Transaction Date"),
            "Initial Quantity": row.get("Quantity"),
            "Current Quantity": row.get("Quantity"),
            "Buy Price": row.get("Transaction Price"),
            "Buy Fee": row.get("Fee"),
            "Sell Fee": 0.0,
            "Realized PnL": 0.0,
            "Status": "OPEN",
        })
    elif row.get("Transaction Type") == "SELL":
        quantity_to_sell = row.get("Quantity")

        for lot in lots:
            if quantity_to_sell == 0:
                break

            if lot["Ticker"] == row.get("Ticker") and lot["Status"] == "OPEN":
                if lot["Current Quantity"] >= quantity_to_sell:
                    quantity_sold = quantity_to_sell
                else:
                    quantity_sold = lot["Current Quantity"]

                pro_rata_ratio = quantity_sold / row.get("Quantity")
                pro_rata_sell_fee = row.get("Fee") * pro_rata_ratio

                pro_rata_buy_fee = lot['Buy Fee'] * (quantity_sold / lot['Initial Quantity'])

                lot["Current Quantity"] -= quantity_sold
                lot["Sell Fee"] += pro_rata_sell_fee

                income = quantity_sold * row.get("Transaction Price")
                cost = quantity_sold * lot["Buy Price"]
                lot["Realized PnL"] += (income - cost - pro_rata_sell_fee - pro_rata_buy_fee)

                quantity_to_sell -= quantity_sold

                if lot["Current Quantity"] == 0:
                    lot["Status"] = "CLOSED"

pd.DataFrame(lots)

,Ticker,Buy Date,Initial Quantity,Current Quantity,Buy Price,Buy Fee,Sell Fee,Realized PnL,Status
0,VUSA.L,2022-08-26,1,1,66.480,0.00,0.0,0.0,OPEN
1,VUSA.L,2022-09-14,1,1,65.158,0.00,0.0,0.0,OPEN
2,VUSA.L,2022-10-18,1,1,62.430,0.00,0.0,0.0,OPEN
3,VUSA.L,2022-11-15,2,2,63.878,0.00,0.0,0.0,OPEN
4,VUSA.L,2022-12-16,2,2,60.163,0.00,0.0,0.0,OPEN
...,...,...,...,...,...,...,...,...,...
67,VUAA.L,2026-03-12,9,9,129.920,2.15,0.0,0.0,OPEN
68,VUAA.L,2026-04-20,4,4,136.520,2.07,0.0,0.0,OPEN
69,VUAA.L,2026-05-12,2,2,142.300,2.03,0.0,0.0,OPEN
70,VUAA.L,2026-06-11,4,4,141.400,2.07,0.0,0.0,OPEN
